# Once Upon a Matchmaker, in Python

In the game, the **Matchmaker** matches people with films the way a dating site matches people with people.
Every customer fills in a profile on three axes, real or fiction, comedy or drama, action or calm, and a film
critic describes every film on the same three axes. Turned into numbers, the profiles and the critic's answers
are the **embeddings**: three numbers for each customer and three for each film. The score of a customer for a
film is the inner product of their two rows, and all the scores together make the table of scores, the
**rating matrix**.

The notebook has two parts. Do each part when its game sends you here, at the end of the game:

1. **Part 1, multiplying two matrices**, after part one of the Matchmaker, matrix multiplication: one score,
   then one row, then the whole table of scores (about 30 minutes).
2. **Part 2, the predictions and the losses**, after part two of the Matchmaker, matrix factorization: the
   predictions of a guess, the loss of every rating, and their average (about 20 minutes).

Parts 3 and 4 go with the optional game, gradient descent up close, and are in a second notebook, also
optional: [the optional notebook in Colab](https://colab.research.google.com/github/eth-ainit-hs26/w2-cx-public/blob/main/02_cx_matrix_factorization_optional_student.ipynb).

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A
folded cell with a title holds plumbing: run it, and open it only if you are curious. The 💡 hints are folded
cells too: double-click one to open it, and open the next one only if you are still stuck. After each task, a
check cell tells you whether your code works.

# Part 1. Multiplying two matrices

In part one of the game the Matchmaker filled its whole table of scores from the profiles, one inner product for
each customer and each film, and the operation got its name: **matrix multiplication**. Now you teach it to a
computer, in small steps:

- **Task A**: one score, Leo and *Men in Black*;
- **Task B**: one row, a customer's scores for every film;
- **Task C**: the whole table, which is matrix multiplication;
- **Task D**: the rule that says when two matrices can be multiplied.

At the end your code multiplies the bigger pair of scene 4. Run the setup cell first.

In [ ]:
#@title Setup of part 1 (run this first) { display-mode: "form" }
# The plumbing of part 1: show_matrix, which draws a matrix as a table, and the checks of tasks A to D.
# You do not need to read it. A check prints one short message and never stops the notebook.
try:
    from IPython import get_ipython as _p1_get_ipython
    from IPython.display import display as _p1_display
except ImportError:
    _p1_get_ipython = None
    _p1_display = None

# The game's numbers, kept here as well, so that the checks still work if you change the lists in the notebook
_P1_CUSTOMER_MATRIX = [[2, 1, -2], [-2, 2, 2], [1, -2, 1]]
_P1_FILM_MATRIX = [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]
_P1_CUSTOMER_NAMES = ['Sara', 'Leo', 'Maya']
_P1_FILM_NAMES = ['The Intouchables', 'Apollo 13', 'Men in Black', 'Interstellar']
_P1_AXES = ['real', 'comedy', 'action']
_P1_BIG_A = [[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]]
_P1_BIG_B = [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]
_P1_BIG_PRODUCT = [[-3, 6, 4, -3, 1, -2], [3, -4, 5, -2, 6, -2], [-2, 5, -3, 3, -3, 0], [4, -4, -3, 3, -2, 3], [0, 5, 4, -2, 0, 0]]
_P1_ONE = (1, 2)   # task A: Leo's row in customer_matrix, and the row of Men in Black in film_matrix


def _p1_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p1_and(items):
    """a, b and c"""
    items = [str(x) for x in items]
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


def _p1_count(n, word):
    """1 row, 3 rows"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _p1_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


# show_matrix: a matrix as a small table

def _p1_entry(x):
    """One entry of a table as text: 7, -2, 0.75 (two decimals for a decimal number), ? for None."""
    if x is None:
        return "?"
    if x is Ellipsis:
        return "..."
    if isinstance(x, bool):
        return str(x)
    if isinstance(x, float):
        return "%.2f" % x
    return str(x)


def _p1_tint(x):
    """The background of an entry, as in the game: blue above 0, orange below 0, grey for 0, and a stronger
    tint from 2 and -2 on. The tints are see-through, so the table reads in a light and in a dark theme."""
    if not _p1_is_number(x):
        return ""
    if x > 0:
        colour = "59,130,214"
    elif x < 0:
        colour = "224,116,42"
    else:
        return "background:rgba(140,140,140,0.16);"
    strength = 0.16
    if abs(x) >= 2:
        strength = 0.32
    return "background:rgba(%s,%.2f);" % (colour, strength)


def _p1_not_a_table(matrix):
    """None when matrix is a table (a list of rows of the same length, each entry a single value), else a pair
    (kind, reason): kind "wait" when a task is not finished yet, "shape" when the value has another shape."""
    if matrix is None:
        return ("wait", "there is no table yet, only None (does the function end with a return line?)")
    if matrix is Ellipsis:
        return ("wait", "the table is still ..., a gap in a 🎯 cell")
    if not isinstance(matrix, (list, tuple)):
        return ("shape", "this is a single value, %s, and a table is a list of rows" % _p1_short(matrix))
    if len(matrix) == 0:
        return ("wait", "the table is an empty list, []")
    for r in range(len(matrix)):
        row = matrix[r]
        if row is Ellipsis:
            return ("wait", "row %d of the table is still ..., a gap in a 🎯 cell" % r)
        if row is None:
            return ("wait", "row %d of the table is None (does the function that makes a row end with a return "
                            "line?)" % r)
        if not isinstance(row, (list, tuple)):
            return ("shape", "row %d is %s, a single value, and each row of a table is a list of numbers"
                    % (r, _p1_short(row)))
    lengths = []
    for row in matrix:
        lengths.append(len(row))
    if max(lengths) == 0:
        if len(matrix) == 1:
            return ("wait", "its only row is an empty list, []")
        return ("wait", "its rows are empty lists, []")
    if min(lengths) != max(lengths):
        return ("shape", "its rows have different lengths, %s, and the rows of a table are all equally long"
                % _p1_and(lengths))
    for row in matrix:
        for x in row:
            if x is Ellipsis:
                return ("wait", "it still holds a ..., a gap in a 🎯 cell")
            if isinstance(x, (list, tuple)):
                return ("shape", "its entries are lists, and each entry of a table is a single number")
    return None


def _p1_labels(names, n):
    """The labels of n rows or columns: the names when there is one name for each, the numbers from 0 when
    names is None, and None when the names do not fit."""
    if names is None:
        return [str(i) for i in range(n)]
    if isinstance(names, (list, tuple)) and len(names) == n:
        return [str(x) for x in names]
    return None


def show_matrix(matrix, row_names=None, column_names=None):
    """Show a matrix (a list of rows) as a table, with a name for each row and each column. Without names,
    the rows and the columns are numbered from 0, the way Python counts. When matrix is not a table yet
    (None, a gap, an empty list, rows of different lengths), it prints one calm line and stops nothing."""
    try:
        problem = _p1_not_a_table(matrix)
        if problem is not None:
            kind, reason = problem
            if kind == "wait":
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % reason)
            else:
                print("⏳ This is not a table yet: %s. The check of your function says more." % reason)
            return
        rows = len(matrix)
        columns = len(matrix[0])
        size = "%s and %s" % (_p1_count(rows, "row"), _p1_count(columns, "column"))
        row_labels = _p1_labels(row_names, rows)
        column_labels = _p1_labels(column_names, columns)
        if row_labels is None or column_labels is None:
            swapped = (isinstance(row_names, (list, tuple)) and isinstance(column_names, (list, tuple))
                       and len(row_names) == columns and len(column_names) == rows and rows != columns)
            if swapped:
                print("Note: this table has %s, and the names are for %s and %s. Are its rows and columns "
                      "swapped? They are numbered below." % (size, _p1_count(columns, "row"),
                                                              _p1_count(rows, "column")))
            else:
                print("Note: the names do not fit this table of %s, so it is numbered below." % size)
            row_labels = [str(i) for i in range(rows)]
            column_labels = [str(j) for j in range(columns)]

        entries = []
        for r in range(rows):
            line = []
            for c in range(columns):
                line.append(_p1_entry(matrix[r][c]))
            entries.append(line)

        # Plain text, for screens that cannot show the table
        name_width = max(len(label) for label in row_labels)
        widths = []
        for c in range(columns):
            width = len(column_labels[c])
            for r in range(rows):
                width = max(width, len(entries[r][c]))
            widths.append(width)
        lines = [" " * name_width + "".join("   " + column_labels[c].rjust(widths[c]) for c in range(columns))]
        for r in range(rows):
            lines.append(row_labels[r].ljust(name_width)
                         + "".join("   " + entries[r][c].rjust(widths[c]) for c in range(columns)))
        lines.append(size)
        text = "\n".join(line.rstrip() for line in lines)

        # The same table in HTML, each number tinted by its sign
        from html import escape
        edge = "border:1px solid rgba(128,128,128,0.3);"
        html = '<table style="border-collapse:collapse;font-size:15px;margin:6px 0 2px 0">'
        html += '<tr><th style="padding:4px 8px"></th>'
        for c in range(columns):
            html += '<th style="padding:4px 10px;text-align:center;font-weight:600">%s</th>' % escape(column_labels[c])
        html += "</tr>"
        for r in range(rows):
            html += '<tr><th style="padding:4px 12px 4px 4px;text-align:left;font-weight:600">%s</th>' % escape(
                row_labels[r])
            for c in range(columns):
                html += ('<td style="padding:5px 12px;min-width:2.2em;text-align:center;'
                         'font-variant-numeric:tabular-nums;%s%s">%s</td>'
                         % (edge, _p1_tint(matrix[r][c]), escape(entries[r][c])))
            html += "</tr>"
        html += "</table>"
        html += '<div style="font-size:13px;opacity:0.75;margin:0 0 8px 2px">%s</div>' % escape(size)

        shell = None
        if _p1_get_ipython is not None:
            shell = _p1_get_ipython()
        if _p1_display is None or shell is None:
            print(text)
        else:
            _p1_display({"text/plain": text, "text/html": html}, raw=True)
    except Exception as error:
        print("The table could not be shown (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_on_its_side(matrix):
    """The matrix turned on its side (its transpose): row j of the result is column j of matrix."""
    try:
        turned = []
        for j in range(len(matrix[0])):
            new_row = []
            for i in range(len(matrix)):
                new_row.append(matrix[i][j])
            turned.append(new_row)
        return turned
    except Exception:
        return None


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P1_TESTS = {
    "inner_product": [
        (([2, 1, -2], [2, 1, -1]), 7,
         '',
         ''),
        (([-2, 2, 2], [2, -2, 1]), -6,
         '',
         ''),
        (([2, 1, -2], [2, -2, 1]), 0,
         '',
         ''),
        (([1, -2, 0, 2], [1, 1, 0, -1]), -3,
         '',
         ''),
        (([2], [-2]), -4,
         '',
         ''),
    ],
    "scores_for_customer": [
        (([2, 1, -2], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [7, 0, -7, -6],
         'scores_for_customer(customer_matrix[0], film_matrix)',
         "Sara's score for each of the four films, in the order of film_matrix"),
        (([-2, 2, 2], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [-4, -6, 10, 0],
         'scores_for_customer(customer_matrix[1], film_matrix)',
         "Leo's score for each of the four films, in the order of film_matrix"),
        (([1, -2, 1], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [-1, 7, -2, 2],
         'scores_for_customer(customer_matrix[2], film_matrix)',
         "Maya's score for each of the four films, in the order of film_matrix"),
        (([2, 1, -2], [[-2, -2, 0], [2, 1, -1]]), [-6, 7],
         'scores_for_customer([2, 1, -2], [[-2, -2, 0], [2, 1, -1]])',
         'Sara with two films only, Interstellar first and The Intouchables second: one score for each, in that order'),
        (([-2, 2, 2], [[-2, 1, 2]]), [10],
         'scores_for_customer([-2, 2, 2], [[-2, 1, 2]])',
         'With one film the answer is a list that holds one score'),
        (([1, -2, 0, 2], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), [-3, 6, 4, -3, 1, -2],
         'scores_for_customer(big_a[0], big_b)',
         'big_a and big_b are the bigger pair of scene 4 (you meet them at the end of part 1): a row of 4 numbers and a matrix of 6 rows give 6 scores'),
    ],
    "matrix_multiply": [
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [[7, 0, -7, -6], [-4, -6, 10, 0], [-1, 7, -2, 2]],
         'matrix_multiply(customer_matrix, film_matrix)',
         "The Matchmaker's table: one row per customer, one column per film"),
        (([[2, 1, -2]], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), [[7, 0, -7, -6]],
         'matrix_multiply([[2, 1, -2]], film_matrix)',
         "A matrix with one row, Sara's, times the films: a table with one row"),
        (([[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]], [[2, 1, -2], [-2, 2, 2], [1, -2, 1]]), [[7, -4, -1], [0, -6, 7], [-7, 10, -2], [-6, 0, 2]],
         'matrix_multiply(film_matrix, customer_matrix)',
         'The other way round: one row per film, one column per customer'),
        (([[3]], [[-2]]), [[-6]],
         'matrix_multiply([[3]], [[-2]])',
         'Two matrices of one row and one column: a table of one row that holds one number'),
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[2, 1, -2], [-2, 2, 2], [1, -2, 1]]), [[9, -6, -2], [-6, 12, -4], [-2, -4, 6]],
         'matrix_multiply(customer_matrix, customer_matrix)',
         'A matrix times itself: every customer with every customer'),
        (([[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), [[-3, 6, 4, -3, 1, -2], [3, -4, 5, -2, 6, -2], [-2, 5, -3, 3, -3, 0], [4, -4, -3, 3, -2, 3], [0, 5, 4, -2, 0, 0]],
         'matrix_multiply(big_a, big_b)',
         'The bigger pair of scene 4 (you meet it at the end of part 1)'),
    ],
    "can_multiply": [
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[2, 1, -1], [2, -2, 1], [-2, 1, 2], [-2, -2, 0]]), True,
         'can_multiply(customer_matrix, film_matrix)',
         'Both have 3 columns, one per axis. Their numbers of rows, 3 and 4, play no part in the rule'),
        (([[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), True,
         'can_multiply(big_a, big_b)',
         'Both have 4 columns: the bigger pair of scene 4, which you meet at the end of part 1'),
        (([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[1, 0], [0, 1], [1, 1], [2, 0]]), False,
         'can_multiply([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[1, 0], [0, 1], [1, 1], [2, 0]])',
         'Rows of 3 numbers and rows of 2 numbers: the third number has no partner'),
        (([[1, 2]], [[3, 4], [5, 6], [7, 8]]), True,
         'can_multiply([[1, 2]], [[3, 4], [5, 6], [7, 8]])',
         'One row and three rows, 2 columns each'),
        (([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4]]), False,
         'can_multiply([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4]])',
         'Two rows each, with 3 columns and 2 columns'),
        (([[2, 1, -2], [-2, 2, 2], [1, -2, 1]], [[1, 1, 0, -1], [0, -2, 2, 1], [2, 0, -1, 1], [-1, 1, 1, 0], [1, 2, -2, 2], [0, -1, 1, -2]]), False,
         'can_multiply(customer_matrix, big_b)',
         '3 columns and 4 columns'),
        (([[1, 2, 0]], [[1, 0], [0, 1], [2, 2]]), False,
         'can_multiply([[1, 2, 0]], [[1, 0], [0, 1], [2, 2]])',
         '3 columns and 2 columns, although the second matrix has 3 rows'),
    ],
}

_P1_HINTS = {'scores_for_customer': 'in the loop, score = inner_product(customer_row, film_matrix[j]), then scores.append(score).', 'matrix_multiply': 'row i of the product is scores_for_customer(matrix_a[i], matrix_b): the inner products of matrix_a[i] with every row of matrix_b.', 'can_multiply': 'columns_b = len(matrix_b[0]); then, if columns_a == columns_b, return True, else return False.'}
_P1_SUCCESS = {'one_score': "Task A works: Leo's row [-2, 2, 2] and the row of Men in Black [-2, 1, 2] give (-2)*(-2) + 2*1 + 2*2 = 4 + 2 + 4 = 10.", 'scores_for_customer': "Sara's scores for the four films are [7, 0, -7, -6].", 'matrix_multiply': "It gives the Matchmaker's table of scores, 3 rows and 4 columns, and the bigger product of scene 4, 5 rows and 6 columns.", 'can_multiply': 'True for the customers and the films, 3 columns each; False for 3 columns and 2 columns.', 'big_product': 'big_product has 5 rows and 6 columns and the same 30 numbers as scene 4 of the game.'}
_P1_COUNTING = {'customer_matrix': "Python counts from 0: customer_matrix[0] is Sara's row, customer_matrix[1] Leo's and customer_matrix[2] Maya's.", 'film_matrix': 'Python counts from 0: film_matrix[0] is the row of The Intouchables, film_matrix[1] of Apollo 13, film_matrix[2] of Men in Black and film_matrix[3] of Interstellar.'}


def _p1_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced.
    Gives back (answer, error), where error is None when the call went through. Never raises."""
    import contextlib, copy, io
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copy.deepcopy(inputs))
        return answer, None
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def _p1_right(name, answer, expected):
    """True when answer is the expected one; can_multiply must answer True or False themselves."""
    try:
        if name == "can_multiply":
            return isinstance(answer, bool) and answer == expected
        return bool(answer == expected)
    except Exception:
        return False


def _p1_first_failure(name, fn):
    """The first test of name that fn fails, or None when fn passes them all."""
    for inputs, expected, call, why in _P1_TESTS[name]:
        answer, error = _p1_call(fn, inputs)
        if error is not None or not _p1_right(name, answer, expected):
            return {"inputs": inputs, "answer": answer, "error": error, "expected": expected, "call": call,
                    "why": why}
    return None


def _p1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function, so no source is needed; a gap that stands on
    # its own line is only visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p1_rows_of(value):
    """True when value is a list of lists."""
    return isinstance(value, list) and len(value) > 0 and isinstance(value[0], list)


def _p1_explain_row(failure):
    """(what went wrong, hint) for scores_for_customer."""
    answer, expected = failure["answer"], failure["expected"]
    customer_row, film_matrix = failure["inputs"]
    error = failure["error"]
    if error is not None:
        if "IndexError" in error:
            return (None, "an index went past the end of a list. In the loop, j counts the films: "
                          "film_matrix[j] is the row of film j, and customer_row is the customer's whole row.")
        if "TypeError" in error:
            return (None, "inner_product takes two rows, each a list of numbers: customer_row and "
                          "film_matrix[j], the row of one film. film_matrix alone is the whole matrix.")
        return (None, _P1_HINTS["scores_for_customer"])
    if answer is None:
        return (None, "end the function with return scores.")
    if not isinstance(answer, list):
        return ("gave %s, expected the list %r" % (_p1_short(answer), expected),
                "return the list scores, with one score per film. A single number is the score of one film only.")
    if len(answer) == 0:
        return ("gave an empty list, expected %r" % expected,
                "scores is still empty: in each round of the loop, put the score at the end of the list with "
                "scores.append(score).")
    for x in answer:
        if isinstance(x, (list, tuple)):
            return ("gave a list of lists, %s, expected %r" % (_p1_short(answer), expected),
                    "each entry of scores is one number, a score. Did you append the row of a film, "
                    "film_matrix[j], in place of its score?")
    if len(answer) != len(expected):
        return ("gave %s, %s, expected %d, %r" % (_p1_count(len(answer), "score"), _p1_short(answer),
                                                  len(expected), expected),
                "one score per row of film_matrix: append exactly one score in each round of the loop, inside the "
                "loop.")
    if len(answer) > 1 and answer.count(answer[0]) == len(answer) and expected.count(expected[0]) != len(expected):
        return ("gave %r, expected %r" % (answer, expected),
                "every score is the same: in round j of the loop, use the row of film j, film_matrix[j].")
    sara = []
    for film_row in film_matrix:
        sara.append(_p1_inner(_P1_CUSTOMER_MATRIX[0], film_row))
    if answer == sara and customer_row != _P1_CUSTOMER_MATRIX[0]:
        return ("gave %r, expected %r" % (answer, expected),
                "these are %s's scores: use customer_row, the row that the function gets, in place of "
                "customer_matrix[0]." % _P1_CUSTOMER_NAMES[0])
    return ("gave %r, expected %r" % (answer, expected), _P1_HINTS["scores_for_customer"])


def _p1_explain_product(answer, matrix_a, matrix_b, expected):
    """(what went wrong, hint) for a product that differs from the expected one. answer is not None."""
    rows, columns = len(expected), len(expected[0])
    wanted = "%s, one for each row of matrix_a, and %s, one for each row of matrix_b" % (
        _p1_count(rows, "row"), _p1_count(columns, "column"))
    if not isinstance(answer, list):
        return ("gave %s; expected a table of %s" % (_p1_short(answer), wanted),
                "return product, the list of rows.")
    if len(answer) == 0:
        return ("gave an empty list; expected a table of %s" % wanted,
                "product is still empty: append each new_row to product, inside the loop.")
    flat = True
    for row in answer:
        if not _p1_is_number(row):
            flat = False
    if flat:
        return ("gave a list of %s, %s; expected a table of %s" % (_p1_count(len(answer), "number"),
                                                                   _p1_short(answer), wanted),
                "the product is a list of rows. new_row is a whole row, the scores of matrix_a[i] with every "
                "row of matrix_b, and product.append(new_row) adds it to the table.")
    for r in range(len(answer)):
        row = answer[r]
        if row is None:
            return ("gave a table whose row %d is None" % r,
                    "new_row is None: does scores_for_customer end with return scores?")
        if not isinstance(row, list):
            return ("gave a table whose row %d is %s, a single value" % (r, _p1_short(row)),
                    "each row of the product is a list of numbers, one for each row of matrix_b.")
    for row in answer:
        for x in row:
            if isinstance(x, (list, tuple)):
                return ("gave a table whose entries are lists, such as %s" % _p1_short(x),
                        "each entry of the product is one number, an inner product. Put numbers into new_row: "
                        "appending a whole list puts a list inside the row.")
    # The Matchmaker's table again, although the function got other matrices: it reads the notebook's lists
    if (matrix_a, matrix_b) != (_P1_CUSTOMER_MATRIX, _P1_FILM_MATRIX):
        matchmaker = []
        for customer_row in _P1_CUSTOMER_MATRIX:
            scores = []
            for film_row in _P1_FILM_MATRIX:
                scores.append(_p1_inner(customer_row, film_row))
            matchmaker.append(scores)
        if answer == matchmaker:
            return ("gave the Matchmaker's table of scores again; expected a table of %s" % wanted,
                    "use matrix_a and matrix_b, the two matrices that the function gets, in place of "
                    "customer_matrix and film_matrix.")
    # Row i paired with the wrong matrix: scores_for_customer(matrix_b[i], matrix_a)
    if len(matrix_b) >= len(matrix_a) and len(matrix_a) != len(matrix_b):
        turned_call = []
        for i in range(len(matrix_a)):
            scores = []
            for row_a in matrix_a:
                scores.append(_p1_inner(matrix_b[i], row_a))
            turned_call.append(scores)
        if answer == turned_call:
            return ("gave a table of %s and %s; expected %s" % (_p1_count(len(answer), "row"),
                                                                _p1_count(len(answer[0]), "column"), wanted),
                    "the two matrices changed places: row i of the product is scores_for_customer(matrix_a[i], "
                    "matrix_b), with a row of matrix_a first and the whole matrix_b second.")
    lengths = []
    for row in answer:
        lengths.append(len(row))
    growing = []
    for r in range(len(answer)):
        growing.append(columns * (r + 1))
    one_list = True
    for row in answer:
        if row is not answer[0]:
            one_list = False
    if len(answer) > 1 and (lengths == growing or (one_list and lengths[0] != columns)):
        return ("gave rows of %s numbers; expected %s" % (_p1_and(lengths), wanted),
                "new_row keeps growing: start a new, empty row, new_row = [], for each row of matrix_a, inside "
                "the outer loop.")
    if min(lengths) != max(lengths):
        return ("gave rows of different lengths, %s; expected %s" % (_p1_and(lengths), wanted),
                "every row of the product has one number for each row of matrix_b.")
    got_rows, got_columns = len(answer), lengths[0]
    if (got_rows, got_columns) != (rows, columns):
        got = "a table of %s and %s" % (_p1_count(got_rows, "row"), _p1_count(got_columns, "column"))
        if (got_rows, got_columns) == (columns, rows):
            return ("gave %s; expected %s" % (got, wanted),
                    "rows and columns are swapped. The outer loop goes over the rows of matrix_a, and each new "
                    "row holds one inner product for each row of matrix_b.")
        if got_rows != rows:
            return ("gave %s; expected %s" % (got, wanted),
                    "the product has one row for each row of matrix_a: append one new_row in each round of the "
                    "loop over the rows of matrix_a.")
        return ("gave %s; expected %s" % (got, wanted),
                "each row of the product holds one number for each row of matrix_b: the inner products of "
                "matrix_a[i] with matrix_b[0], matrix_b[1] and so on.")
    # The right size, some wrong numbers
    flat_rows = True
    for r in range(rows):
        if answer[r].count(answer[r][0]) != columns:
            flat_rows = False
    expected_flat = True
    for r in range(rows):
        if expected[r].count(expected[r][0]) != columns:
            expected_flat = False
    if columns > 1 and flat_rows and not expected_flat:
        return ("has the right size, %s and %s, but every number in a row is the same: row 0 is %r, expected %r"
                % (_p1_count(rows, "row"), _p1_count(columns, "column"), answer[0], expected[0]),
                "the entry in row i and column j pairs matrix_a[i] with matrix_b[j], a different row of matrix_b "
                "for each column. Check which index goes with which matrix.")
    for i in range(rows):
        for j in range(columns):
            if answer[i][j] != expected[i][j]:
                return ("has the right size, %s and %s, but the entry in row %d and column %d is %s, expected %r"
                        % (_p1_count(rows, "row"), _p1_count(columns, "column"), i, j, _p1_short(answer[i][j]),
                           expected[i][j]),
                        "the entry in row %d and column %d is the inner product of matrix_a[%d], %r, with "
                        "matrix_b[%d], %r." % (i, j, i, matrix_a[i], j, matrix_b[j]))
    return ("gave %s" % _p1_short(answer), _P1_HINTS["matrix_multiply"])


def _p1_explain_columns(fn, failure):
    """(what went wrong, hint) for can_multiply, from the pattern of its answers on all the tests."""
    answer, error = failure["answer"], failure["error"]
    if error is not None:
        if "IndexError" in error:
            return (None, "use the first row of each matrix, matrix_a[0] and matrix_b[0]: every matrix has one, "
                          "even a matrix with a single row.")
        return (None, "len gives the length of a list: len(matrix_a[0]) is the number of columns of matrix_a.")
    if answer is None:
        return (None, "end with a return line: if columns_a == columns_b, return True, else return False.")
    if not isinstance(answer, bool):
        if answer in ("True", "False"):
            return ("gave the text %r, expected %r" % (answer, failure["expected"]),
                    "return True or False without quotes: they are Python's own values for yes and no.")
        return ("gave %s, expected %r" % (_p1_short(answer), failure["expected"]),
                "return True or False: compare the two numbers of columns with == inside an if.")
    answers, expected, rows, columns_rows, rows_columns = [], [], [], [], []
    for inputs, wanted, call, why in _P1_TESTS["can_multiply"]:
        matrix_a, matrix_b = inputs
        answers.append(_p1_call(fn, inputs)[0])
        expected.append(wanted)
        rows.append(len(matrix_a) == len(matrix_b))
        columns_rows.append(len(matrix_a[0]) == len(matrix_b))
        rows_columns.append(len(matrix_a) == len(matrix_b[0]))
    what = "gave %r, expected %r" % (answer, failure["expected"])
    if answers == rows:
        return (what, "you compared the numbers of rows, len(matrix_a) and len(matrix_b). The rule is about the "
                      "columns: compare the lengths of the rows, len(matrix_a[0]) and len(matrix_b[0]).")
    if answers == columns_rows:
        return (what, "you compared the columns of matrix_a with the number of rows of matrix_b, len(matrix_b). "
                      "Compare the columns of both: len(matrix_a[0]) and len(matrix_b[0]).")
    if answers == rows_columns:
        return (what, "you compared the number of rows of matrix_a, len(matrix_a), with the columns of matrix_b. "
                      "Compare the columns of both: len(matrix_a[0]) and len(matrix_b[0]).")
    if answers == [True] * len(answers):
        return (what, "your function says True every time, also for a pair that cannot be multiplied. Return "
                      "False when columns_a and columns_b differ (else: return False).")
    if answers == [False] * len(answers):
        return (what, "your function never says True. Return True when columns_a == columns_b.")
    flipped = []
    for x in expected:
        flipped.append(not x)
    if answers == flipped:
        return (what, "True and False are swapped: return True when the numbers of columns are equal.")
    return (what, _P1_HINTS["can_multiply"])


def _p1_explain(name, fn, failure):
    """(what went wrong, hint) for the first test that fn fails."""
    if name == "scores_for_customer":
        return _p1_explain_row(failure)
    if name == "can_multiply":
        return _p1_explain_columns(fn, failure)
    if failure["error"] is not None:
        error = failure["error"]
        if "IndexError" in error:
            return (None, "an index went past the end of a matrix. i counts the rows of matrix_a, so row i is "
                          "matrix_a[i]; inside, j counts the rows of matrix_b: pair matrix_a[i] with matrix_b[j].")
        if "TypeError" in error:
            return (None, "inner_product takes two rows, such as matrix_a[i] and matrix_b[j]; "
                          "scores_for_customer takes one row and a whole matrix, such as matrix_a[i] and matrix_b.")
        return (None, _P1_HINTS["matrix_multiply"])
    if failure["answer"] is None:
        return (None, "end the function with return product.")
    matrix_a, matrix_b = failure["inputs"]
    return _p1_explain_product(failure["answer"], matrix_a, matrix_b, failure["expected"])


def _p1_check(name, fn, needs):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    failure = _p1_first_failure(name, fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P1_TESTS[name]), _P1_SUCCESS[name]))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, "
                    "run its cell, then run this check again." % name)
    if _p1_has_gap(fn):
        print(not_finished)
        return
    for other, place in needs:
        if not _p1_uses(fn, other):
            continue
        other_fn = globals().get(other)
        if other_fn is None or _p1_first_failure(other, other_fn) is not None:
            if other == "inner_product":
                print("⏳ %s uses inner_product, which does not work any more. Run the cell of inner_product "
                      "again (it is given, so undo any change to it), then run this check again." % name)
            else:
                print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, "
                      "then run this check again." % (name, other, place, place))
            return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    what, hint = _p1_explain(name, fn, failure)
    if failure["error"] is not None:
        print("❌ %s stopped with an error: %s" % (failure["call"], failure["error"]))
    elif failure["answer"] is None:
        print("❌ %s gave no answer (None). Does the function end with a return line?" % failure["call"])
    else:
        print("❌ %s %s. %s." % (failure["call"], what, failure["why"]))
    print("   💡 Hint: " + hint)


def check_scores_for_customer(fn):
    """Task B: tests scores_for_customer on the three customers, on fewer films and on the bigger pair."""
    try:
        _p1_check("scores_for_customer", fn, [("inner_product", "the given cell")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def check_matrix_multiply(fn):
    """Task C: tests matrix_multiply on six pairs of matrices, from 1 x 1 to the bigger pair of scene 4."""
    try:
        _p1_check("matrix_multiply", fn, [("inner_product", "the given cell"),
                                          ("scores_for_customer", "task B")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def check_can_multiply(fn):
    """Task D: tests can_multiply on seven pairs of matrices, some that fit and some that do not."""
    try:
        _p1_check("can_multiply", fn, [])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_which_row(value, matrix):
    """The index of value among the rows of matrix, or -1."""
    for i in range(len(matrix)):
        try:
            if value == matrix[i]:
                return i
        except Exception:
            pass
    return -1


def _p1_row_problem(value, variable, matrix_name, matrix, names, wanted, other_name, other_matrix):
    """None when value is row number wanted of matrix, else (what went wrong, hint), for task A."""
    if _p1_which_row(value, [matrix[wanted]]) == 0:
        return None
    k = _p1_which_row(value, matrix)
    if k >= 0:
        return ("%s is %s, the row of %s, %s[%d]" % (variable, _p1_short(value), names[k], matrix_name, k),
                _P1_COUNTING[matrix_name])
    if _p1_which_row(value, other_matrix) >= 0:
        return ("%s is %s, a row of %s" % (variable, _p1_short(value), other_name),
                "the row of %s is in %s." % (names[wanted], matrix_name))
    if _p1_is_number(value):
        return ("%s is %s, a single number" % (variable, _p1_short(value)),
                "a row is a whole list of numbers, and %s[%d] picks one. A second index, as in %s[%d][0], "
                "would pick one number out of that row." % (matrix_name, wanted, matrix_name, wanted))
    if _p1_rows_of(value):
        return ("%s is a whole matrix" % variable,
                "pick one row out of it with an index in square brackets, such as %s[0] for the first row."
                % matrix_name)
    return ("%s is %s, and the row of %s is a list of %d numbers" % (variable, _p1_short(value), names[wanted],
                                                                     len(matrix[wanted])),
            _P1_COUNTING[matrix_name])


def _p1_check_one(leo_row, film_row, score):
    gaps = []
    for variable, value in (("leo_row", leo_row), ("men_in_black_row", film_row), ("leo_and_men_in_black", score)):
        if value is Ellipsis:
            gaps.append(variable)
    if gaps:
        verb = "holds"
        if len(gaps) > 1:
            verb = "hold"
        print("⏳ Task A is not finished yet: %s still %s ... (Python calls it Ellipsis). Replace each ... with "
              "your code, run the cell, then run this check again." % (_p1_and(gaps), verb))
        return
    c, f = _P1_ONE
    problem = _p1_row_problem(leo_row, "leo_row", "customer_matrix", _P1_CUSTOMER_MATRIX, _P1_CUSTOMER_NAMES, c,
                              "film_matrix", _P1_FILM_MATRIX)
    if problem is None:
        problem = _p1_row_problem(film_row, "men_in_black_row", "film_matrix", _P1_FILM_MATRIX, _P1_FILM_NAMES, f,
                                  "customer_matrix", _P1_CUSTOMER_MATRIX)
    if problem is None:
        expected = _p1_inner(_P1_CUSTOMER_MATRIX[c], _P1_FILM_MATRIX[f])
        if not _p1_is_number(score):
            problem = ("leo_and_men_in_black is %s, and a score is one number" % _p1_short(score),
                       "leo_and_men_in_black = inner_product(leo_row, men_in_black_row).")
        elif score != expected:
            fn = globals().get("inner_product")
            if fn is None or _p1_first_failure("inner_product", fn) is not None:
                print("⏳ inner_product does not work any more. Run its cell again (it is given, so undo any change "
                      "to it), then the cell of task A, then this check.")
                return
            problem = ("leo_and_men_in_black is %s, expected %d" % (_p1_short(score), expected),
                       "let inner_product do the work: leo_and_men_in_black = inner_product(leo_row, "
                       "men_in_black_row).")
    if problem is None:
        print("✅ " + _P1_SUCCESS["one_score"])
        return
    print("❌ %s." % problem[0])
    print("   💡 Hint: " + problem[1])


def check_one_score(leo_row, men_in_black_row, leo_and_men_in_black):
    """Task A: Leo's row, the row of Men in Black, and their score."""
    try:
        _p1_check_one(leo_row, men_in_black_row, leo_and_men_in_black)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_check_big(product):
    for name, mine in (("big_a", _P1_BIG_A), ("big_b", _P1_BIG_B)):
        if name in globals() and globals()[name] != mine:
            print("❌ %s is no longer the matrix of scene 4. Run the cell with big_a and big_b again, then the "
                  "cells below it." % name)
            return
    problem = _p1_not_a_table(product)
    if product is None or product is Ellipsis or (problem is not None and problem[0] == "wait"):
        print("⏳ big_product is not ready yet: your matrix_multiply of task C is not finished. Get its check to "
              "✅, then run the cells of the bigger pair again.")
        return
    if product == _P1_BIG_PRODUCT:
        print("✅ " + _P1_SUCCESS["big_product"])
        return
    what, hint = _p1_explain_product(product, _P1_BIG_A, _P1_BIG_B, _P1_BIG_PRODUCT)
    print("❌ big_product %s." % what.replace("gave ", "is ", 1))
    print("   💡 Hint: " + hint + " The check of task C tells you more.")


def check_big_product(big_product):
    """The bigger pair: compares big_product with the product of scene 4 of the game."""
    try:
        _p1_check_big(big_product)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


print("✅ Setup of part 1 complete.")

## The two matrices of the game

The Matchmaker's numbers as Python lists. A matrix is a list of rows. `customer_matrix` has one row per customer,
and each row holds three numbers, one per axis, in the order of `axes`. `film_matrix` has one row per film, on the
same three axes. Python counts from 0: `customer_matrix[0]` is Sara's row, `[2, 1, -2]`, and `film_matrix[0]` is the row
of *The Intouchables*, `[2, 1, -1]`. The customers, their profiles and the critic are invented for the lesson.

In [ ]:
# The Matchmaker's data, from part one of the game (invented for the lesson)
axes = ["real", "comedy", "action"]
customer_names = ["Sara", "Leo", "Maya"]
film_names = ["The Intouchables", "Apollo 13", "Men in Black", "Interstellar"]

# The customers' profiles: one row per customer, one number per axis
customer_matrix = [
    [ 2,  1, -2],   # Sara
    [-2,  2,  2],   # Leo
    [ 1, -2,  1],   # Maya
]

# The critic's answers: one row per film, on the same three axes
film_matrix = [
    [ 2,  1, -1],   # The Intouchables (2011)
    [ 2, -2,  1],   # Apollo 13 (1995)
    [-2,  1,  2],   # Men in Black (1997)
    [-2, -2,  0],   # Interstellar (2014)
]

In [ ]:
print("customer_matrix, the profiles of the customers:")
show_matrix(customer_matrix, customer_names, axes)
print("film_matrix, the answers of the critic:")
show_matrix(film_matrix, film_names, axes)

## The inner product, given

You wrote this function yourself in the video store notebook: multiply two rows position by position, then add
up the products. Here it is, ready to use. Run its cell, and the next one, which uses it once.

In [ ]:
# Given: the inner product of two rows, the function you wrote in the video store notebook
def inner_product(row_a, row_b):
    # row_a and row_b are two lists of numbers of the same length.
    # Multiply them position by position, then add up the products.
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total

In [ ]:
# Sara is row 0 of customer_matrix, and The Intouchables is row 0 of film_matrix
print("Sara and The Intouchables:", inner_product(customer_matrix[0], film_matrix[0]))

**Reading the result.** 7, as in scene 2 of the game. Sara's row `[2, 1, -2]` and the row of *The Intouchables*, `[2, 1, -1]`: the products,
axis by axis, are 4, 1 and 2, and their sum is 4 + 1 + 2 = 7. They agree on all three axes. In Python:
`2*2 + 1*1 + (-2)*(-1) = 4 + 1 + 2 = 7`.

## 🎯 Task A. One score

What does Leo think of *Men in Black*? Pick the two rows out of the matrices by their index, then let `inner_product`
compute the score. Remember that Python counts from 0: Sara is customer 0, and *The Intouchables* is film 0. In the cell
below, replace each `...` with your code and run it.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Python counts the rows of a matrix from 0: customer_matrix[0] is the first row, Sara's.
# Leo is the second customer. Which number picks his row?
# Men in Black is the third film in film_matrix. Which number picks its row?

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
# leo_row = customer_matrix[1]
# men_in_black_row = film_matrix[...]                  # the third film: count 0, 1, 2
# leo_and_men_in_black = inner_product(leo_row, men_in_black_row)

In [ ]:
# 🎯 YOUR TURN, task A: one score, Leo and Men in Black
leo_row = ...                # 🎯 YOUR TURN: Leo's row of customer_matrix, picked by its index
men_in_black_row = ...       # 🎯 YOUR TURN: the row of Men in Black in film_matrix
leo_and_men_in_black = ...   # 🎯 YOUR TURN: the inner product of the two rows

print("Leo's row:           ", leo_row)
print("Men in Black's row:  ", men_in_black_row)
print("Leo and Men in Black:", leo_and_men_in_black)

In [ ]:
check_one_score(leo_row, men_in_black_row, leo_and_men_in_black)

## 🎯 Task B. One row

A customer's row of the table of scores holds four scores, one for each film. Write
`scores_for_customer(customer_row, film_matrix)`. It gets the row of one customer and the matrix of the films, and
returns a list with the customer's score for every film, in the order of the films. For Sara,
`scores_for_customer(customer_matrix[0], film_matrix)` gives `[7, 0, -7, -6]`.

The loop is given: `for j in range(len(film_matrix))` runs once for each film, with j equal to 0, 1, 2 and 3, so
`film_matrix[j]` is the row of film j. A list grows with `append`: after `scores = []` and `scores.append(7)`,
`scores` is `[7]`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The loop runs once for each film: j is 0, then 1, 2 and 3, and film_matrix[j] is the row of film j.
# In each round, the score is the inner product of customer_row with film_matrix[j].
# Then put that score at the end of the list: scores.append(score).

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     for j in range(len(film_matrix)):
#         score = inner_product(customer_row, film_matrix[j])
#         ...                                   # put score at the end of the list scores

In [ ]:
# 🎯 YOUR TURN, task B: one customer's scores for every film
def scores_for_customer(customer_row, film_matrix):
    # customer_row is the row of one customer; film_matrix has one row per film.
    # Return a list with the customer's score for every film, in the order of the films.
    scores = []
    for j in range(len(film_matrix)):
        # film_matrix[j] is the row of film number j
        score = ...   # 🎯 YOUR TURN: the customer's score for film j (use inner_product)
        ...           # 🎯 YOUR TURN: put score at the end of the list scores
    return scores

In [ ]:
check_scores_for_customer(scores_for_customer)

In [ ]:
# Sara's scores, shown as a table with one row: a list that holds the list of her scores
sara_scores = scores_for_customer(customer_matrix[0], film_matrix)
show_matrix([sara_scores], ["Sara"], film_names)

## 🎯 Task C. The whole table

The table of scores has one row per customer, and each row is what your `scores_for_customer` computes. So the
whole table is a loop around your loop: for every row of the first matrix, its scores with every row of the
second. That is **matrix multiplication**, as in scene 3 of the game. Two matrices with the same number of
columns are multiplied by taking the inner product of every row of the first with every row of the second. The
product has one row for each row of the first matrix and one column for each row of the second: its entry in row
i and column j is the inner product of row i of the first with row j of the second.

Write `matrix_multiply(matrix_a, matrix_b)`. It returns the product as a list of rows. You may assume that the
two matrices have the same number of columns. `matrix_multiply(customer_matrix, film_matrix)` should give the
Matchmaker's table of scores, 3 rows and 4 columns.

💭 **Think first.** How many inner products does the whole table take?

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# Read matrix_a as the customers and matrix_b as the films.
# Row i of the product is then customer i's scores for every film:
# the inner products of matrix_a[i] with every row of matrix_b.
# In task B you wrote a function that computes exactly one such row.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# scores_for_customer(customer_row, film_matrix) from task B makes one row of the product.
# Here the customer's row is matrix_a[i], and the matrix of the films is matrix_b.

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#     for i in range(len(matrix_a)):
#         new_row = scores_for_customer(matrix_a[i], ...)
#         product.append(new_row)
#
# You may also write the loop of task B again, inside this loop:
#         new_row = []
#         for j in range(len(matrix_b)):
#             new_row.append(inner_product(..., matrix_b[j]))
#         product.append(new_row)

In [ ]:
# 🎯 YOUR TURN, task C: the whole table, matrix multiplication
def matrix_multiply(matrix_a, matrix_b):
    # matrix_a and matrix_b have the same number of columns. The product has one row for
    # each row of matrix_a and one column for each row of matrix_b: its row i holds the
    # inner products of matrix_a[i] with every row of matrix_b.
    product = []
    for i in range(len(matrix_a)):
        new_row = ...   # 🎯 YOUR TURN: row i of the product, the scores of matrix_a[i]
        product.append(new_row)
    return product

In [ ]:
check_matrix_multiply(matrix_multiply)

In [ ]:
# The whole table of scores: the product of the customers' matrix and the films' matrix
show_matrix(matrix_multiply(customer_matrix, film_matrix), customer_names, film_names)

**Reading the result.** Twelve inner products, one for each cell: 3 customers times 4 films. Each customer has a
clear favourite: Sara *The Intouchables* (7), Leo *Men in Black* (10) and Maya *Apollo 13* (7). Two cells are 0: for Sara and *Apollo 13* the products 4, -2 and -2 cancel out, and so do 4, -4 and 0 for Leo and *Interstellar*.
These are the twelve scores of the game, and your `matrix_multiply` computed them all from the two matrices.
Inside it, `scores_for_customer` runs a loop over the films, so matrix multiplication is two loops, one inside
the other: every row of the first matrix with every row of the second.

## 🎯 Task D. The column rule

An inner product pairs two rows position by position, so the two rows must be equally long. That is why two
matrices can be multiplied only when they have the same number of columns: then each number in a row of the
first has a partner in every row of the second. The numbers of rows can differ, as with 3 customers and 4
films.

💭 **Think first.** Can a matrix of 3 rows and 3 columns be multiplied with a matrix of 4 rows and 2 columns?

Write `can_multiply(matrix_a, matrix_b)`. It returns `True` when the two matrices have the same number of
columns, and `False` otherwise. The number of columns of a matrix is the length of one of its rows:
`len(matrix_a[0])` is the length of the first row of `matrix_a`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# The number of columns of a matrix is the length of one of its rows.
# columns_a = len(matrix_a[0]) measures the first row of matrix_a: do the same for matrix_b.
# Then compare the two numbers with == inside an if.

In [ ]:
#@title 💡 Hint 2 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#     columns_b = len(matrix_b[0])
#     if columns_a == columns_b:
#         return True
#     else:
#         return ...

In [ ]:
# 🎯 YOUR TURN, task D: the column rule
def can_multiply(matrix_a, matrix_b):
    # Return True when matrix_a and matrix_b have the same number of columns, and False otherwise.
    columns_a = len(matrix_a[0])   # the number of columns of matrix_a: the length of its first row
    columns_b = ...                # 🎯 YOUR TURN: the number of columns of matrix_b
    # 🎯 YOUR TURN: if columns_a and columns_b are equal, return True; else return False
    ...

In [ ]:
check_can_multiply(can_multiply)

In [ ]:
# The think-first question: a matrix of 3 rows and 3 columns, and one of 4 rows and 2 columns
three_by_three = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
four_by_two = [[1, 0], [0, 1], [1, 1], [2, 0]]
print("Can they be multiplied?", can_multiply(three_by_three, four_by_two))

**Reading the result.** `False`: a row of the first matrix has 3 numbers and a row of the second only 2, so the
third number has no partner. The 4 rows of the second matrix play no part in the rule.

## A bigger product

Scene 4 of the game multiplied a bigger pair, `big_a` with 5 rows and 4 columns and `big_b` with 6 rows and 4 columns: plain invented numbers this
time, with no customers and no films. Without names, `show_matrix` numbers the rows and the columns from 0, the
way Python counts. The game numbered them from 1, so row 0 here is row 1 in the game.

In [ ]:
# The bigger pair of scene 4 of the game: plain invented numbers, 4 columns each
big_a = [
    [ 1, -2,  0,  2],
    [ 2,  1, -1,  0],
    [-1,  0,  2,  1],
    [ 0,  2,  1, -2],
    [ 2, -1,  1,  1],
]

big_b = [
    [ 1,  1,  0, -1],
    [ 0, -2,  2,  1],
    [ 2,  0, -1,  1],
    [-1,  1,  1,  0],
    [ 1,  2, -2,  2],
    [ 0, -1,  1, -2],
]

print("big_a:")
show_matrix(big_a)
print("big_b:")
show_matrix(big_b)

💭 **Think first.** Can `big_a` and `big_b` be multiplied? How many rows and columns will their product have? And
can `customer_matrix` be multiplied with `big_b`? Then run the cell, and the check below it.

In [ ]:
print("Can big_a and big_b be multiplied?", can_multiply(big_a, big_b))
print("Can customer_matrix and big_b be multiplied?", can_multiply(customer_matrix, big_b))

big_product = matrix_multiply(big_a, big_b)
show_matrix(big_product)

In [ ]:
check_big_product(big_product)

**Reading the result.** 5 rows, one for each row of `big_a`, and 6 columns, one for each row of `big_b`: 30
inner products of 4 products each, the same table as in scene 4 of the game. The entry in row 2 and column 3 (row 3
and column 4 in the game) is the inner product of `big_a[2]` and `big_b[3]`: `(-1)*(-1) + 0*1 + 2*1 + 1*0 = 1 + 0 + 2 + 0 = 3`. `customer_matrix` and `big_b` cannot be multiplied: their
rows have 3 and 4 numbers.

## Optional: the same product in textbooks

Textbooks, and the Python libraries you may meet later, write matrix multiplication with one change: they pair
the rows of the first matrix with the **columns** of the second. To get the Matchmaker's table that way, they
first turn the film matrix on its side, so that each film becomes a column. The turned matrix is called the
**transpose**. Every inner product stays the same, and so does the whole product. Their rule, "the columns of the
first match the rows of the second", is the column rule of this notebook, applied after the turn. The folded
cell below shows the film matrix turned on its side.

In [ ]:
#@title ▶ Optional: the film matrix turned on its side { display-mode: "form" }
print("film_matrix turned on its side: one column per film, one row per axis")
show_matrix(_p1_on_its_side(film_matrix), axes, film_names)

## Back to the game

That is part 1: your `matrix_multiply` fills the Matchmaker's whole table from the profiles. In real life nobody
fills in a profile, and the Matchmaker sees only the ratings. Part two of the Matchmaker, matrix factorization,
starts from the ratings and looks for the numbers behind them. Part 2 of this notebook comes after it.

# Part 2. The predictions and the losses

In part two of the game the Matchmaker knows only the ratings: nobody fills in a profile. It guesses an
embedding for every customer and every film, two numbers each, and measures how good the guess is: the
predictions, the loss of every rating, and their average. In this part you teach the computer the same
measurement:

- **Task A**: the predictions of a guess, with your `matrix_multiply` from part 1;
- **Task B**: the loss of one rating;
- **Task C**: the loss of every rating;
- **Task D**: the average loss, the whole guess in one number.

Run the setup cell of part 2 first.

If you come back after a break, Colab may have forgotten part 1. Then click on the setup cell below and choose
**Runtime > Run before** in Colab's menu: it runs every cell above it, part 1 included. Then run the setup
cell.

In [ ]:
#@title Setup of part 2 (run this first) { display-mode: "form" }
# The plumbing of part 2: print_loss, which prints one loss in a line, and the checks of tasks A to D.
# You do not need to read it. A check prints one short message and never stops the notebook.

# Part 2 uses three things from part 1: inner_product, matrix_multiply and show_matrix. When part 1 has not run in
# this session, this cell defines them, so that part 2 works on its own.
_p2_missing = []
if "inner_product" not in globals():
    _p2_missing.append("inner_product")

    def inner_product(row_a, row_b):
        # row_a and row_b are two lists of numbers of the same length.
        # Multiply them position by position, then add up the products.
        total = 0
        for i in range(len(row_a)):
            total = total + row_a[i] * row_b[i]
        return total

if "matrix_multiply" not in globals():
    _p2_missing.append("matrix_multiply")

    def matrix_multiply(matrix_a, matrix_b):
        # Every row of matrix_a with every row of matrix_b: row i of the product holds the inner products
        # of matrix_a[i] with each row of matrix_b.
        product = []
        for i in range(len(matrix_a)):
            new_row = []
            for j in range(len(matrix_b)):
                new_row.append(inner_product(matrix_a[i], matrix_b[j]))
            product.append(new_row)
        return product

if "show_matrix" not in globals():
    _p2_missing.append("show_matrix")

    def show_matrix(matrix, row_names=None, column_names=None):
        """A plain stand-in for the show_matrix of part 1: the table as text, one line per row."""
        try:
            problem = _p2_not_a_table(matrix)
            if problem is not None:
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % problem)
                return
            rows, columns = len(matrix), len(matrix[0])
            if not isinstance(row_names, (list, tuple)) or len(row_names) != rows:
                row_names = [str(i) for i in range(rows)]
            if not isinstance(column_names, (list, tuple)) or len(column_names) != columns:
                column_names = [str(j) for j in range(columns)]
            entries = [[_p2_text(matrix[i][j]) for j in range(columns)] for i in range(rows)]
            width = max(len(str(name)) for name in row_names)
            widths = []
            for j in range(columns):
                w = len(str(column_names[j]))
                for i in range(rows):
                    w = max(w, len(entries[i][j]))
                widths.append(w)
            print(" " * width + "".join("   " + str(column_names[j]).rjust(widths[j]) for j in range(columns)))
            for i in range(rows):
                print(str(row_names[i]).ljust(width)
                      + "".join("   " + entries[i][j].rjust(widths[j]) for j in range(columns)))
        except Exception as error:
            print("The table could not be shown (%s)." % error)


def _p2_short(value):
    """A value as text, cut short when it is long; a gap as words."""
    if value is Ellipsis:
        return "... (a gap in a 🎯 cell)"
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p2_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _p2_text(x):
    """A number as text: 25, or at most 4 decimals: 10.6667, 0.25."""
    if not _p2_is_number(x):
        return _p2_short(x)
    if isinstance(x, float) and x == int(x) and abs(x) < 1e15:
        return "%d" % x
    if isinstance(x, float):
        if x != 0 and abs(x) < 0.0001:
            return "%.2g" % x
        return ("%.4f" % x).rstrip("0").rstrip(".")
    return str(x)


def _p2_rows(table):
    """A table as text: [[-3, 4], [2, 4], [2, -6]]."""
    try:
        return "[" + ", ".join("[" + ", ".join(_p2_text(x) for x in row) + "]" for row in table) + "]"
    except Exception:
        return _p2_short(table)


def _p2_waiting(value):
    """Why value is not a number yet, in words."""
    if value is Ellipsis:
        return "it is still ..., a gap in a 🎯 cell"
    if value is None:
        return "it is None (does the function end with a return line?)"
    return "it is %s" % _p2_short(value)


def _p2_not_a_table(matrix):
    """None when matrix is a table of numbers (rows of the same length), else the reason in words."""
    if matrix is Ellipsis:
        return "the table is still ..., a gap in a 🎯 cell"
    if matrix is None:
        return "there is no table yet, only None (does the function end with a return line?)"
    if not isinstance(matrix, (list, tuple)) or len(matrix) == 0:
        return "%s is not a table of numbers" % _p2_short(matrix)
    for row in matrix:
        if row is Ellipsis:
            return "a row is still ..., a gap in a 🎯 cell"
        if not isinstance(row, (list, tuple)):
            return "a row is %s, and each row of a table is a list of numbers" % _p2_short(row)
        if len(row) == 0:
            return "its rows are empty lists, []"
        for x in row:
            if x is Ellipsis:
                return "it still holds a ..., a gap in a 🎯 cell"
            if not _p2_is_number(x):
                return "it holds %s where a number belongs" % _p2_short(x)
    if len(set(len(row) for row in matrix)) > 1:
        return "its rows have different lengths"
    return None


# print_loss: one loss in a line

def print_loss(text, loss):
    """Print the text and a loss, at most 4 decimals, for example
    The average loss of the first guess: 10.6667
    While a 🎯 task is not finished it prints a calm line instead."""
    try:
        if not _p2_is_number(loss):
            print("%s: ⏳ no loss yet, %s. The check of the 🎯 task above says more." % (text, _p2_waiting(loss)))
            return
        print("%s: %s" % (text, _p2_text(loss)))
    except Exception:
        pass


# The arithmetic, kept here for the checks

def _p2_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p2_product(a, b):
    return [[_p2_inner(r, s) for s in b] for r in a]


def _p2_loss(r, p):
    return (r - p) * (r - p)


def _p2_losses(ratings, predictions):
    return [[_p2_loss(ratings[i][j], predictions[i][j]) for j in range(len(ratings[i]))] for i in range(len(ratings))]


def _p2_flat_losses(ratings, predictions):
    out = []
    for row in _p2_losses(ratings, predictions):
        out.extend(row)
    return out


def _p2_total(table):
    total = 0
    for row in table:
        for x in row:
            total = total + x
    return total


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P2_TESTS = {
    "predict": [
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, 2], [-2, -1]]), [[-3, 4], [2, 4], [2, -6]],
         'predict([[-1, -2], [-2, 0], [2, 2]], [[-1, 2], [-2, -1]])',
         "The Matchmaker's first guess: Ida's row [-1, -2] with the row of Paddington, [-1, 2], gives 1 + (-4) = -3, and the same for the other five pairs"),
        (([[2]], [[3]]), [[6]],
         'predict([[2]], [[3]])',
         'One customer and one film with one number each: 2 times 3 is 6'),
        (([[1, 2]], [[3, 4], [5, 6]]), [[11, 17]],
         'predict([[1, 2]], [[3, 4], [5, 6]])',
         'One customer and two films: one row with two predictions, 3 + 8 and 5 + 12'),
        (([[1, 0], [0, 1]], [[2, 3]]), [[2], [3]],
         'predict([[1, 0], [0, 1]], [[2, 3]])',
         'Two customers and one film: one column, a prediction for each customer'),
        (([[2, 1, -2]], [[2, 1, -1], [2, -2, 1]]), [[7, 0]],
         'predict([[2, 1, -2]], [[2, 1, -1], [2, -2, 1]])',
         'Three numbers per row: Sara and two films of part 1, 4 + 1 + 2 = 7 and 4 - 2 - 2 = 0'),
        (([[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]]), [[2, 1], [-1, 2], [1, -2]],
         'predict([[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]])',
         "A guess whose predictions are the ratings: the films' rows [1, 0] and [0, 1] pick each customer's first and second number"),
        (([[0.5, -1]], [[2, 0.5]]), [[0.5]],
         'predict([[0.5, -1]], [[2, 0.5]])',
         'Decimals: 0.5 times 2 plus -1 times 0.5 is 1 - 0.5 = 0.5'),
    ],
    "squared_loss": [
        ((2, -3), 25,
         'squared_loss(2, -3)',
         "Ida's rating of Paddington, 2, and the first guess's prediction, -3, differ by 5, and 5 times 5 is 25"),
        ((2, 2), 0,
         'squared_loss(2, 2)',
         'A prediction that matches the rating misses by 0, and its loss is 0'),
        ((2, 0), 4,
         'squared_loss(2, 0)',
         'A miss of 2 costs 2 times 2, which is 4'),
        ((2, 4), 4,
         'squared_loss(2, 4)',
         'A prediction 2 too high: the difference is -2, and -2 times -2 is 4'),
        ((-1, 2), 9,
         'squared_loss(-1, 2)',
         'The rating -1 and the prediction 2 differ by -3, and -3 times -3 is 9'),
        ((1, 1), 0,
         'squared_loss(1, 1)',
         'The rating 1 and the prediction 1: no miss, no loss'),
        ((2, 1.5), 0.25,
         'squared_loss(2, 1.5)',
         'A prediction with decimals: the difference is 0.5, and 0.5 times 0.5 is 0.25'),
    ],
    "loss_matrix": [
        (([[2, 1], [-1, 2], [1, -2]], [[-3, 4], [2, 4], [2, -6]]), [[25, 9], [9, 4], [1, 16]],
         'loss_matrix([[2, 1], [-1, 2], [1, -2]], [[-3, 4], [2, 4], [2, -6]])',
         "The first guess: Ida's rating of Paddington, 2, and its prediction, -3, differ by 5, and 5 times 5 is 25; the same for the other five ratings"),
        (([[2, 1], [-1, 2], [1, -2]], [[2, 1], [-1, 2], [1, -2]]), [[0, 0], [0, 0], [0, 0]],
         'loss_matrix([[2, 1], [-1, 2], [1, -2]], [[2, 1], [-1, 2], [1, -2]])',
         'Predictions equal to the ratings miss by 0, so every loss is 0'),
        (([[1]], [[3]]), [[4]],
         'loss_matrix([[1]], [[3]])',
         'One rating, 1, and one prediction, 3: the difference -2, and -2 times -2 is 4'),
        (([[2, -1, 0]], [[0, 0, 0]]), [[4, 1, 0]],
         'loss_matrix([[2, -1, 0]], [[0, 0, 0]])',
         'One customer, three films, every prediction 0: the losses 4, 1 and 0'),
        (([[1], [2]], [[0], [0]]), [[1], [4]],
         'loss_matrix([[1], [2]], [[0], [0]])',
         'Two customers and one film: the losses 1 and 4, one in each row'),
        (([[2, 1]], [[1.5, 1.25]]), [[0.25, 0.0625]],
         'loss_matrix([[2, 1]], [[1.5, 1.25]])',
         'Decimals: the differences 0.5 and -0.25, the losses 0.25 and 0.0625'),
    ],
    "average": [
        (([[25, 9], [9, 4], [1, 16]],), 10.666666666666666,
         'average([[25, 9], [9, 4], [1, 16]])',
         'The six losses of the first guess add up to 64, and 64 / 6 is 10.6667'),
        (([[4]],), 4.0,
         'average([[4]])',
         'One loss alone: the average is the loss itself'),
        (([[1, 2, 3]],), 2.0,
         'average([[1, 2, 3]])',
         'One row of three losses: 6 / 3 is 2'),
        (([[1], [2], [6]],), 3.0,
         'average([[1], [2], [6]])',
         'One column of three losses: 9 / 3 is 3'),
        (([[0, 0], [0, 0], [0, 0]],), 0.0,
         'average([[0, 0], [0, 0], [0, 0]])',
         'Six losses of 0: the average is 0'),
        (([[1, 2], [3, 4], [5, 6], [7, 8]],), 4.5,
         'average([[1, 2], [3, 4], [5, 6], [7, 8]])',
         'Four rows of two losses: 36 / 8 is 4.5'),
        (([[0.25, 0.5]],), 0.375,
         'average([[0.25, 0.5]])',
         'Decimals: 0.75 / 2 is 0.375'),
    ],
}

_P2_SUCCESS = {'predict': 'The first guess gives the six predictions of the board: -3 and 4 for Ida, 2 and 4 for Raj, 2 and -6 for Elsa.', 'squared_loss': "Ida's rating 2 and the prediction -3 differ by 5, and the loss is 25.", 'loss_matrix': 'The first guess gives the six losses of the board: 25 and 9 for Ida, 9 and 4 for Raj, 1 and 16 for Elsa.', 'average': 'The six losses of the first guess add up to 64, and 64 / 6 is 10.6667, 10.67 on the board.'}
_P2_FIRST_PREDICTIONS = [[-3, 4], [2, 4], [2, -6]]


def _p2_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced. Gives back (answer,
    error, the copies after the call); error is None when the call went through. Never raises."""
    import contextlib, copy, io
    copies = copy.deepcopy(inputs)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copies)
        return answer, None, copies
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error), copies


def _p2_same(answer, expected):
    """True when answer is expected: a number within a tiny margin, or a table of such numbers."""
    try:
        if _p2_is_number(expected):
            return _p2_is_number(answer) and abs(answer - expected) < 1e-9
        if isinstance(expected, (list, tuple)):
            if not isinstance(answer, (list, tuple)) or len(answer) != len(expected):
                return False
            for a, e in zip(answer, expected):
                if not _p2_same(a, e):
                    return False
            return True
        return bool(answer == expected)
    except Exception:
        return False


def _p2_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function; a gap that stands on its own line is only
    # visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p2_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p2_flat(value):
    """The numbers inside a value of nested lists and tuples."""
    if isinstance(value, (list, tuple)):
        out = []
        for x in value:
            out.extend(_p2_flat(x))
        return out
    return [value]


def _p2_slip(fn, name, slips):
    """The hint of the first slip that fn repeats on every whole-number test of name, or None. A slip that stops
    with an error on a test matches only when fn stops with an error there too."""
    for slip, hint in slips:
        same = True
        tested = 0
        for inputs, expected, call, why in _P2_TESTS[name]:
            if not all(_p2_is_number(x) and x == int(x) for x in _p2_flat(inputs)):
                continue
            answer, error, after = _p2_call(fn, inputs)
            try:
                wanted, raises = slip(*inputs), False
            except Exception:
                wanted, raises = None, True
            tested += 1
            if raises:
                if error is None:
                    same = False
                    break
            elif error is not None or not _p2_same(answer, wanted):
                same = False
                break
        if same and tested > 0:
            return hint
    return None


def _p2_error_hint(error, name):
    """A hint for an error that stopped a participant's function."""
    if "NameError" in error:
        return ("Python does not know a name that %s uses. Run the setup cell of part 2 and the cells of the tasks "
                "above, then the cell of %s, then this check." % (name, name))
    if "unsupported operand" in error and "^" in error:
        return "in Python ^ means something else than a power. Multiply the difference by itself: difference * difference."
    if name == "predict":
        return ("predict gives its two matrices to matrix_multiply, the customers first: "
                "matrix_multiply(customer_embeddings, film_embeddings).")
    if name == "squared_loss":
        return ("squared_loss gets two numbers: difference = rating - prediction, then loss = difference * "
                "difference, then return loss.")
    if name == "loss_matrix":
        if "IndexError" in error:
            return ("ratings[i][j] and predictions[i][j] use the same i and j, in this order: i is the row, a "
                    "customer, and j the column, a film.")
        return ("in row i and column j: loss = squared_loss(ratings[i][j], predictions[i][j]), then row.append(loss) "
                "puts it at the end of the row.")
    if name == "average":
        if "ZeroDivisionError" in error:
            return "count is still 0 after the loops: add 1 to it for each loss, count = count + 1."
        return "inside the loops, total = total + losses[i][j] and count = count + 1; after them, return total / count."
    return "run the setup cell of part 2 again, then the cell of your function, then this check."


# Task A, predict

_P2_PREDICT_SLIPS = [
    (lambda c, f: _p2_product(f, c),
     "this puts the films first, so the table has one row per film. The predictions have one row per customer and "
     "one column per film: matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: _p2_product(c, c),
     "this multiplies the customers' matrix with itself. The second matrix is the films': "
     "matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: _p2_product(f, f),
     "this multiplies the films' matrix with itself. The predictions pair the customers with the films: "
     "matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: c,
     "this is the customers' matrix itself. The predictions are its product with the films' matrix: "
     "matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: _P2_FIRST_PREDICTIONS,
     "the predictions must come from the two matrices that the function gets. Your code gave the predictions of "
     "the first guess every time: matrix_multiply(customer_embeddings, film_embeddings)."),
]


def _p2_run_predict(fn):
    """None when predict passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["predict"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "predict"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return predictions?" % call,
                    "the last line is return predictions.")
        if _p2_is_number(answer):
            return ("%s gave one number, %s, and the predictions are a table, %s. %s." % (
                call, _p2_text(answer), _p2_rows(expected), why),
                "one number is one prediction. predict returns the table of all of them, one row per customer: "
                "matrix_multiply(customer_embeddings, film_embeddings).")
        hint = _p2_slip(fn, "predict", _P2_PREDICT_SLIPS)
        if hint is None:
            hint = ("the predictions are the product of the two matrices, the customers first: "
                    "matrix_multiply(customer_embeddings, film_embeddings).")
        return ("%s gave %s, expected %s. %s." % (call, _p2_rows(answer), _p2_rows(expected), why), hint)
    return None


# Task B, squared_loss

_P2_LOSS_SLIPS = [
    (lambda r, p: r - p, "this is the difference, the rating minus the prediction. The loss is the difference "
                         "times itself: loss = difference * difference."),
    (lambda r, p: p - r, "this is the prediction minus the rating, and it is not squared yet. Take the rating "
                         "minus the prediction, then multiply that difference by itself."),
    (lambda r, p: 2 * (r - p), "times 2 doubles the difference. The loss multiplies the difference by itself: "
                               "difference * difference."),
    (lambda r, p: abs(r - p), "this is the size of the miss. The loss is the difference times itself: "
                              "difference * difference."),
    (lambda r, p: (r - p) ^ 2, "in Python ^ means something else than a power. Multiply the difference by "
                               "itself: difference * difference."),
    (lambda r, p: (r + p) * (r + p), "the difference is the rating minus the prediction: rating - prediction."),
    (lambda r, p: p * p, "this is the prediction times itself. Square the difference: difference = rating - "
                         "prediction, then difference * difference."),
    (lambda r, p: r * r - p * p, "square the difference itself: first difference = rating - prediction, then "
                                 "loss = difference * difference."),
]


def _p2_run_squared_loss(fn):
    """None when squared_loss passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["squared_loss"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "squared_loss"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return loss?" % call,
                    "the last line is return loss.")
        if not _p2_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p2_short(answer), _p2_text(expected), why),
                    "return one number, the loss: the difference times itself.")
        hint = _p2_slip(fn, "squared_loss", _P2_LOSS_SLIPS)
        if hint is None:
            hint = "difference = rating - prediction, then loss = difference * difference, then return loss."
        return ("%s gave %s, expected %s. %s." % (call, _p2_text(answer), _p2_text(expected), why), hint)
    return None


# Task C, loss_matrix

_P2_LOSSES_SLIPS = [
    (lambda r, p: _p2_losses(r, r),
     "this compares each rating with itself, so every loss is 0. Compare the rating with the prediction: "
     "squared_loss(ratings[i][j], predictions[i][j])."),
    (lambda r, p: [[r[i][j] - p[i][j] for j in range(len(r[i]))] for i in range(len(r))],
     "these are the differences, the ratings minus the predictions. The loss is the square of a difference: "
     "squared_loss(ratings[i][j], predictions[i][j])."),
    (lambda r, p: _p2_flat_losses(r, p),
     "this is one flat list of all the losses. The losses are a table with one row per customer: each loss goes "
     "into row, row.append(loss), and the loop over i puts each finished row into losses."),
    (lambda r, p: [_p2_flat_losses(r, p) for i in range(len(r))],
     "every row holds all the losses so far. Start a new row for each customer: row = [] inside the loop over i, "
     "just before the loop over j."),
    (lambda r, p: [[] for i in range(len(r))],
     "the rows are empty: put each loss at the end of its row, row.append(loss)."),
]


def _p2_run_loss_matrix(fn):
    """None when loss_matrix passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["loss_matrix"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "loss_matrix"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return losses?" % call,
                    "the last line is return losses.")
        hint = _p2_slip(fn, "loss_matrix", _P2_LOSSES_SLIPS)
        if hint is None:
            hint = ("in row i and column j: loss = squared_loss(ratings[i][j], predictions[i][j]), then "
                    "row.append(loss).")
        return ("%s gave %s, expected %s. %s." % (call, _p2_short(answer), _p2_rows(expected), why), hint)
    return None


# Task D, average

_P2_AVERAGE_SLIPS = [
    (lambda t: _p2_total(t) / len(t),
     "len(losses) is the number of rows. The count is the number of losses: count = count + 1 for each of them."),
    (lambda t: _p2_total(t) / len(t[0]),
     "len(losses[0]) is the number of columns. The count is the number of losses: count = count + 1 for each of "
     "them."),
    (lambda t: _p2_total(t),
     "this is the total of the losses. The average divides it by their count: return total / count."),
    (lambda t: _p2_total(t) // sum(len(row) for row in t),
     "// divides and rounds down to a whole number. The average divides with /: return total / count."),
    (lambda t: _p2_total(t) / _p2_total(t),
     "count counts the losses: count = count + 1, one more for each loss."),
    (lambda t: _p2_total([t[0]]) / len(t[0]),
     "the return line runs inside the loop over the rows, so the average stops after the first row. Move "
     "return total / count out of both loops, as far left as total = 0."),
]


def _p2_run_average(fn):
    """None when average passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["average"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "average"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line returns the total divided by the count: return total / count.")
        if not _p2_is_number(answer):
            return ("%s gave %s, expected one number, %s. %s." % (call, _p2_short(answer), _p2_text(expected), why),
                    "return one number: the total of the losses divided by their count.")
        hint = _p2_slip(fn, "average", _P2_AVERAGE_SLIPS)
        if hint is None:
            hint = "add every loss to total and 1 to count for each, then return total / count."
        return ("%s gave %s, expected %s. %s." % (call, _p2_text(answer), _p2_text(expected), why), hint)
    return None


_P2_RUNNERS = {"predict": _p2_run_predict, "squared_loss": _p2_run_squared_loss,
               "loss_matrix": _p2_run_loss_matrix, "average": _p2_run_average}
_P2_NEEDS = {"predict": [("matrix_multiply", "part 1")],
             "squared_loss": [],
             "loss_matrix": [("squared_loss", "task B")],
             "average": []}


def _p2_works(name):
    """True when the function called name exists and passes its tests."""
    fn = globals().get(name)
    if not callable(fn):
        return False
    if name == "matrix_multiply":
        for a, b in (([[1, 2]], [[3, 4]]), ([[1, 0], [0, 1]], [[2, 3], [4, 5], [6, 7]]),
                     ([[-1, -2], [-2, 0], [2, 2]], [[-1, 2], [-2, -1]])):
            answer, error, after = _p2_call(fn, (a, b))
            if error is not None or not _p2_same(answer, _p2_product(a, b)):
                return False
        return True
    if _p2_has_gap(fn):
        return False
    return _P2_RUNNERS[name](fn) is None


def _p2_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s. If %s is no "
              "longer a function, run the cell of its task again." % (name, name, name, name, name))
        return
    if _p2_has_gap(fn):
        print("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
              "cell, then run this check again." % name)
        return
    for other, place in _P2_NEEDS[name]:
        if not _p2_uses(fn, other) or _p2_works(other):
            continue
        if other == "matrix_multiply":
            print("⏳ %s uses your matrix_multiply from part 1, which does not work yet. Get the check of task C of "
                  "part 1, check_matrix_multiply(matrix_multiply), to ✅ first, then run this check again." % name)
        else:
            print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, then run "
                  "this check again." % (name, other, place, place))
        return
    failure = _P2_RUNNERS[name](fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P2_TESTS[name]), _P2_SUCCESS[name]))
        return
    print("❌ " + failure[0])
    print("   💡 Hint: " + failure[1])


def check_predict(fn):
    """Task A: tests predict on seven pairs of matrices, from the first guess to one with decimals."""
    try:
        _p2_check("predict", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_squared_loss(fn):
    """Task B: tests squared_loss on seven pairs of a rating and a prediction."""
    try:
        _p2_check("squared_loss", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_loss_matrix(fn):
    """Task C: tests loss_matrix on six pairs of tables, from the first guess to one with decimals."""
    try:
        _p2_check("loss_matrix", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_average(fn):
    """Task D: tests average on seven tables of losses, from the first guess to one with decimals."""
    try:
        _p2_check("average", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


if _p2_missing:
    print("Note: part 1 has not run in this session, so this cell defined %s for part 2. To run part 1 as well, "
          "click on this cell, choose Runtime > Run before in the menu, then run this cell again."
          % (", ".join(_p2_missing[:-1]) + " and " + _p2_missing[-1] if len(_p2_missing) > 1 else _p2_missing[0]))
print("✅ Setup of part 2 complete.")

## The ratings

Three customers, Ida, Raj and Elsa, rated two films, *Paddington* (2014) and *The Martian* (2015), with the five marks of the game turned into
numbers: two ticks 2, a tick 1, a shrug 0, a cross -1, two crosses -2. `ratings` has one row per customer and
one column per film. The customers and their ratings are invented for the lesson.

In [ ]:
# The ratings of part two of the game (invented for the lesson): one row per customer, one column per film
customer_names2 = ["Ida", "Raj", "Elsa"]
film_names2 = ["Paddington", "The Martian"]
ratings = [
    [ 2,  1],   # Ida
    [-1,  2],   # Raj
    [ 1, -2],   # Elsa
]

In [ ]:
show_matrix(ratings, customer_names2, film_names2)

## The first guess

The Matchmaker guesses two numbers for each customer and two for each film. Nobody knows what the two columns
mean, so they have no names: `number_names` calls them the first and the second number. `customer_embeddings`
has one row per customer and `film_embeddings` one row per film. The cell below holds the Matchmaker's first
guess from the board of the game.

In [ ]:
# The Matchmaker's first guess, as on the board of the game: two numbers for each customer and each film
customer_embeddings = [
    [-1, -2],   # Ida
    [-2,  0],   # Raj
    [ 2,  2],   # Elsa
]
film_embeddings = [
    [-1,  2],   # Paddington
    [-2, -1],   # The Martian
]
number_names = ["first number", "second number"]
show_matrix(customer_embeddings, customer_names2, number_names)
show_matrix(film_embeddings, film_names2, number_names)

## 🎯 Task A. The predictions

The prediction of a customer's rating of a film is the inner product of the customer's row and the film's row.
All the predictions together are the product of the two matrices, every customer's row with every film's row:
the matrix multiplication of part 1. Like `ratings`, the predictions have one row per customer and one column
per film.

Write `predict(customer_embeddings, film_embeddings)`. It gets the two matrices and returns the predictions.

💭 **Think first.** Which of the two matrices goes first in the product?

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# The predictions are the product of the two matrices, every customer's row with every film's row.
# Part 1's matrix_multiply(matrix_a, matrix_b) computes exactly that. The rows of matrix_a become
# the rows of the product.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     predictions = matrix_multiply(..., ...)   # the customers first: one row of predictions per customer
#     return predictions

In [ ]:
# 🎯 YOUR TURN, task A: the predictions of a guess
def predict(customer_embeddings, film_embeddings):
    # customer_embeddings has one row per customer, film_embeddings one row per film, with the
    # same number of columns. Return the predictions: one row per customer, one column per film,
    # each the inner product of a customer's row and a film's row.
    predictions = ...   # 🎯 YOUR TURN: the product of the two matrices (use matrix_multiply)
    return predictions

In [ ]:
check_predict(predict)

In [ ]:
# The predictions of the first guess: one row per customer, one column per film
predictions = predict(customer_embeddings, film_embeddings)
show_matrix(predictions, customer_names2, film_names2)

**Reading the result.** The six predictions of the board: -3 and 4 for Ida, 2 and 4 for Raj, 2 and -6 for Elsa. Ida's row `[-1, -2]` and the row of *Paddington*, `[-1, 2]`:
the products are 1 and -4, and their sum is -3. Ida gave *Paddington* 2, so this prediction misses. The customers'
matrix goes first: its rows become the rows of the product, one per customer.

## 🎯 Task B. The loss of one rating

How wrong is a prediction? Its **loss** is the rating minus the prediction, squared. For Ida and *Paddington*:
2 - (-3) = 5, and 5 times 5 is 25.

Write `squared_loss(rating, prediction)`. It gets two numbers, the rating and the prediction, and returns the
loss.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The difference is the rating minus the prediction: for Ida and Paddington, 2 - (-3) = 5.
# In Python: difference = rating - prediction
# The loss is that difference times itself.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     difference = rating - prediction
#     loss = ...                         # the difference times itself, with *
#     return loss

In [ ]:
# 🎯 YOUR TURN, task B: the loss of one rating
def squared_loss(rating, prediction):
    # rating and prediction are two numbers. Return the difference times itself.
    difference = ...   # 🎯 YOUR TURN: the rating minus the prediction
    loss = ...         # 🎯 YOUR TURN: the difference times itself
    return loss

In [ ]:
check_squared_loss(squared_loss)

In [ ]:
# Ida's rating of Paddington, 2, against the first guess's prediction, -3, and three other predictions
print_loss("The loss of the prediction -3", squared_loss(2, -3))
print_loss("The loss of the prediction 1", squared_loss(2, 1))
print_loss("The loss of the prediction 0", squared_loss(2, 0))
print_loss("The loss of the prediction 4", squared_loss(2, 4))

**Reading the result.** 25 for the first guess. A miss of 1 costs 1 and a miss of 2 costs 4: twice the miss,
four times the loss, so bigger mistakes are punished harder. A prediction of 4, 2 too high, costs 4 as well,
because -2 times -2 is 4: too high and too low count alike.

## 🎯 Task C. The loss of every rating

The board of the game shows a loss for every rating, in a table of the same shape as `ratings`. Write
`loss_matrix(ratings, predictions)`. In row `i` and column `j` it puts the loss of the rating `ratings[i][j]`
and the prediction `predictions[i][j]`. The skeleton gives the two loops, over the rows `i` and the columns `j`,
as in part 1: you fill in the loss and put it at the end of its row.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# In row i and column j sit the rating ratings[i][j] and the prediction predictions[i][j].
# squared_loss from task B gives their loss. row.append(loss) puts it at the end of the row.

In [ ]:
#@title 💡 Hint 2 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#             loss = squared_loss(ratings[i][j], ...)
#             row.append(...)

In [ ]:
# 🎯 YOUR TURN, task C: the loss of every rating
def loss_matrix(ratings, predictions):
    # ratings and predictions are two tables of the same shape: one row per customer, one
    # column per film. Return the table of the losses, of the same shape: in row i and
    # column j, the squared loss of ratings[i][j] and predictions[i][j].
    losses = []
    for i in range(len(ratings)):            # row i: one customer
        row = []
        for j in range(len(ratings[i])):     # column j: one film
            loss = ...                        # 🎯 YOUR TURN: the loss in row i, column j (use squared_loss)
            ...                               # 🎯 YOUR TURN: put the loss at the end of row
        losses.append(row)
    return losses

In [ ]:
check_loss_matrix(loss_matrix)

In [ ]:
# The loss of every rating of the first guess
losses = loss_matrix(ratings, predictions)
show_matrix(losses, customer_names2, film_names2)

**Reading the result.** The six losses of the board: 25 and 9 for Ida, 9 and 4 for Raj, 1 and 16 for Elsa. The largest, 25, is Ida's rating of *Paddington*, the
prediction that misses most.

## 🎯 Task D. The average loss

Six losses are hard to compare from one guess to the next. Their **average** sums the whole guess up in one
number: the total of the losses divided by how many there are. The smaller, the better.

Write `average(losses)`. It gets a table of losses and returns the average of all its numbers.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# Visit every loss: add it to total, and add 1 to count. After the two loops,
# total / count is the average. For the first guess: 64 / 6.

In [ ]:
#@title 💡 Hint 2 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#             total = total + losses[i][j]
#             count = ...
#     return total / ...

In [ ]:
# 🎯 YOUR TURN, task D: the average loss
def average(losses):
    # losses is a table, a list of rows of numbers. Return the average of all its numbers:
    # their total divided by how many there are.
    total = 0
    count = 0
    for i in range(len(losses)):
        for j in range(len(losses[i])):
            total = ...   # 🎯 YOUR TURN: the total so far plus losses[i][j]
            count = ...   # 🎯 YOUR TURN: one number more
    return ...            # 🎯 YOUR TURN: the total divided by the count

In [ ]:
check_average(average)

In [ ]:
# The average loss of the first guess
average_loss = average(losses)
print_loss("The average loss of the first guess", average_loss)

**Reading the result.** 10.6667: the six losses add up to 64, and 64 / 6 is 10.666..., which the board of the game
rounds to 10.67. That one number measures the whole guess.

## Back to the game

That is part 2: your `predict`, `loss_matrix` and `average` measure a guess the way the board of the game does,
the measurement behind every small change in the game's rounds. Next is part three of the Matchmaker, the whole
picture. The optional game, gradient descent up close, goes with the optional notebook:
[the optional notebook in Colab](https://colab.research.google.com/github/eth-ainit-hs26/w2-cx-public/blob/main/02_cx_matrix_factorization_optional_student.ipynb).